In [ ]:
"""
Develop/Test case the engine model
"""

import os
from typing import Tuple, Union, Dict, Any, List, Callable
from functools import partial
import traceback

import numpy as np

import jax
import jax.numpy as jnp

import flax.linen as nn
import optax

from tqdm.auto import tqdm

from bng_simulator.utils.logger_utils import load_log_data

In [ ]:
class MLP(nn.Module):
    """ A MLP class
    """
    output_dimension: int
    initial_value_range: float = 0.01
    activation_fn: Union[str, callable] = "tanh"
    layers_archictecture: Tuple = (16, 16)
    fake_normalization: bool = False

    def setup(self):
        # Cache the activation function during setup.
        if isinstance(self.activation_fn, str):
            self._activation_fn = get_activation_fn_from_name(self.activation_fn)
        else:
            self._activation_fn = self.activation_fn
        
        # Create dense layers for each feature dimension.
        self.layers = [
            nn.Dense(
                feat_dimension,
                kernel_init=nn.initializers.uniform(scale=self.initial_value_range)
            )
            for feat_dimension in self.layers_archictecture
        ]
            
        # Create final dense layer.
        self.final_dense = nn.Dense(self.output_dimension)

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        """Forward pass of the MLP
        
        Args:
            x: input to the network (..., n) array
        
        Returns:
            x: output of the network with shape (output_dimension,)
        """
        # Create the first tan-based normalization layer.
        if self.fake_normalization:
            alpha_f = self.param("alpha_f", nn.initializers.ones, ()) * 0.5
            shift_f = self.param("shift_f", nn.initializers.zeros, (x.shape[-1],))
            scale_f = self.param("scale_f", nn.initializers.ones, (x.shape[-1],))
            x = scale_f * jax.nn.tanh(x * alpha_f) + shift_f
            
        # Apply the dense layers.
        for dense in self.layers:
            x = dense(x)
            x = self._activation_fn(x)
        x = self.final_dense(x)
        return x

def get_activation_fn_from_name(
    act_fn_name : str
) -> callable:
    """ Get the activation function from its name.
    
    Args:
        act_fn_name: name of the activation function
            str
    Returns:
        act_fn: the activation function
            callable
    """
    if hasattr(jnp, act_fn_name):
        act_fn = getattr(jnp, act_fn_name)
    else:
        act_fn = getattr(jax.nn, act_fn_name)
    return act_fn

def load_network_from_config(
    name : str,
    **kwargs
) -> nn.Module:
    """ Load a network from a configuration dictionary.
    
    Args:
        name: name of the network to load
            str
        kwargs: dictionary containing the parameters to initialize the
        network structure
            dict
    
    Returns:
        network: the network
            nn.Module
    """
    if name == "MLP":
        return MLP(**kwargs)

    raise ValueError(f"Unknown network name {name}")

In [ ]:
class NNNoiseTerm(nn.Module):
    """
    Template class for a learnable noise term.
    """
    output_dimension: int
    # Mean and std of the state and control variables
    noise_network_config: Dict[str, Any]
    max_noise: float = 1
    ignore_noise: bool = False

    def setup(self):
        nn_type = self.noise_network_config.get("type", "MLP")
        self.noise_network = load_network_from_config(
            nn_type,
            output_dimension=self.output_dimension,
            **self.noise_network_config["args"]
        )

    @nn.compact
    def __call__(self, x: jnp.ndarray, u: jnp.ndarray) -> jnp.ndarray:
        """
        Forward pass of the noise term.
        """
        # If ignore noise, return zeros
        if self.ignore_noise:
            return jnp.zeros(x.shape[-1]), {"max_noise": 0, "noise": jnp.zeros(x.shape[-1])}
        
        # Get the max noise value as parameter
        max_noise = self.param("max_noise", nn.initializers.zeros, (x.shape[-1],))
        max_noise = jnp.exp(max_noise)
        max_noise = jnp.clip(max_noise, 1.0e-4, self.max_noise)
        
        # Normalize the state
        data_state = self.get_normalized_state(x)
        noise = self.noise_network(data_state)
        total_noise = max_noise * (1 - jax.nn.sigmoid(noise)) # Initially some noise
        return total_noise, {"max_noise": max_noise, "noise": noise}

In [ ]:
"""
Let's setup the structure of the engine model
"""
class EngineDynamics(nn.Module):
    """
    Generic engine dynamics module that computes the ODE right-hand side for 
    engine states. State order: [omega_e, p_b, omega_r]
    Control order: [throttle, ...]
    """
    # Configuration dictionaries for each submodule
    net_torque_config: Dict[str, Any]
    boost_config: Dict[str, Any]
    gear_ratio_config: Dict[str, Any]
    guided_parameters: Dict[str, Any]

    # Mean and std of the state and control variables
    state_action_mean: Dict[str, jnp.ndarray]
    state_action_std: Dict[str, jnp.ndarray]

    # State limits
    state_max_min: Dict[str, float]
    use_normalized_input: bool = False

    # latent state configuration
    latent_config: Dict[str, Any] = None
    
    @property
    def name_states(self) -> Tuple[str]:
        return ("engine_speed", "boost_pressure", "rear_wheelspeed")

    @property
    def name_controls(self) -> Tuple[str]:
        return ("throttle",)

    @property
    def num_states(self) -> int:
        return len(self.name_states)
    
    @property
    def num_controls(self) -> int:
        return len(self.name_controls)

    @property
    def name2idx_states(self) -> Dict[str, int]:
        return {name: idx for idx, name in enumerate(self.name_states)}

    @property
    def name2idx_controls(self) -> Dict[str, int]:
        return {name: idx for idx, name in enumerate(self.name_controls)}

    def get_parameter_by_name(self, name: str) -> jnp.ndarray:
        """
        Get a parameter from guided_parameters by name.
        """
        if name not in self.guided_parameters:
            return jnp.exp(self.param(name, nn.initializers.normal(stddev=1), ()))
        init_guess = self.guided_parameters[name]
        scale_factor = self.param(name, nn.initializers.normal(), ())
        return init_guess * jnp.exp(scale_factor)

    def create_networks(
        self,
        config_dict: Dict[str, Any],
        num_out: int = None
    ):
        """
        Temporary function to create a neural network given 
        a configuration.
        """
        if len(config_dict) <= 0:
            return None
        # Construct the neural network
        nn_type = config_dict.get("type", "MLP")

        # Create the network
        if num_out is None:
            return load_network_from_config(
                nn_type,
                **config_dict["args"]
            )

        return load_network_from_config(
            nn_type,
            output_dimension = num_out,
            **config_dict["args"]
        )

    # --- Setup functions for each main parametrization (prototypes) ---
    def setup_net_torque(self):
        """
        Prototype for setting up the net torque equilibrium network.
        """
        pass

    def setup_boost(self):
        """
        Prototype for setting up the boost equilibrium network 
        or boost dynamics parameters.
        """
        pass

    def setup_clutch_gear_ratio(self):
        """
        Prototype for setting up the clucth and gear ratio network.
        """
        pass

    def setup_latent(self):
        """
        Prototype for setting up the latent state network.
        """
        pass

    def calculate_initial_guess_for_latent(
        self,
        state : jnp.ndarray,
        control : jnp.ndarray,
        times: jnp.ndarray
    ) -> jnp.ndarray:
        """
        Calculate the initial guess for the latent state.
        """
        return None

    def calculate_latent_state_field(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        """
        Calculate the latent state.
        """
        return None, {}

    def setup(self):
        """
        Setup the engine dynamics module.
        """
        self.setup_net_torque()
        self.setup_boost()
        self.setup_clutch_gear_ratio()
        self.setup_latent()

    def calculate_net_torque(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        """
        Calculate the net torque equilibrium.
        """
        return 0, {}

    def calculate_boost_field(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        """
        Calculate the derivative for boost pressure p_b. 
        
        For example:
        dot_p_b = (p_b_eq - p_b) / tau_b where p_b_eq is computed 
        from the boost_eq network. This is just one example of models.
        """
        return 0, {}

    def calculate_engine_speed_field(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        """
        Calculate the derivative for engine speed omega_e. 
        
        For example:
        dot_omega_e = (omega_e_eq - omega_e) / tau_omega_e where omega_e_eq is computed 
        from the gear_ratio_eq network. This is just one example of models.
        """
        return 0, {}

    def calculate_clutch_gear_ratio(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        """
        Calculate the gear ratio.
        """
        return 0, {}

    def normalize_state_action(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Dict[str, jnp.ndarray]:
        """
        Normalize the state and action data.
        """
        return {
            name : (val - self.state_action_mean[name]) / self.state_action_std[name] \
                for name, val in data.items()
        }

    def get_normalized_state(
        self,
        state: jnp.ndarray
    ) -> jnp.ndarray:
        """
        Normalize the state.
        """
        normalized_dict = {
            name : (val - self.state_action_mean[name]) / self.state_action_std[name] \
                for name, val in zip(self.name_states, state)
        }
        return jnp.array([normalized_dict[name] for name in self.name_states])
    
    def get_data_from_state_and_control(
        self,
        state: jnp.ndarray,
        control: jnp.ndarray
    ) -> Dict[str, Any]:
        """
        Get the data dictionary from the state and control.
        """
        res_unscaled = {
            **{
                name : state_val for name, state_val in zip(self.name_states, state)
            },
            **{
                name : control_val for name, control_val in zip(self.name_controls, control)
            }
        }
        if self.use_normalized_input:
            res = self.normalize_state_action(res_unscaled)
        else:
            res = res_unscaled
        
        res = {
            **res,
            **{
                f"{name}_unscaled": val for name, val in res_unscaled.items()
            }
        }
        return res

    @property
    def history_size(
        self,
    ):
        """
        Get the history size.
        """
        return 0

    def split_state_with_history(
        self,
        states: jnp.ndarray,
        controls: jnp.ndarray
    ):
        """
        Split the state with history.
        """
        assert states.ndim == 2, "State should be 2D"
        assert controls.ndim == 2, "Controls should be 2D"
        assert states.shape[0] >= self.history_size + 1, \
            f"State shape {states.shape} does not match history size {self.history_size}"
        assert controls.shape[0] > self.history_size, \
            f"Controls shape {controls.shape} does not match history size {self.history_size}"

        if self.history_size == 0:
            return (states[:1], None), (states, controls)
        hist_states = states[:self.history_size+1]
        hist_controls = controls[:self.history_size]
        pred_states = states[self.history_size:]
        pred_controls = controls[self.history_size:]
        return (hist_states, hist_controls), (pred_states, pred_controls)

    def vector_field(
        self,
        state: jnp.ndarray,
        control: jnp.ndarray
    ) -> Tuple[jnp.ndarray, Dict[str, Any]]:
        """
        Combine the individual state derivative calculations to produce the full drift vector.
        Also aggregates auxiliary outputs in an extras dictionary.
        """
        return jnp.zeros_like(state), {}

    def split_state_latent_var(
        self,
        full_state: jnp.ndarray
    ) -> Tuple[jnp.ndarray, jnp.ndarray]:
        """
        Extract the state and latent state from the combined state.
        """
        if self.latent_config is None:
            assert full_state.shape[-1] == self.num_states, \
                f"Full state shape {full_state.shape} does not match num states {self.num_states}"
            return full_state, None
        return full_state[:self.num_states], full_state[self.num_states:]

    def combine_state_and_latent(
        self,
        state: jnp.ndarray,
        latent_state: jnp.ndarray
    ) -> jnp.ndarray:
        """
        Combine the state and latent state.
        """
        if latent_state is None:
            return state
        return jnp.concatenate([state, latent_state])

    @nn.compact
    def __call__(
        self,
        state: jnp.ndarray,
        control: jnp.ndarray
    ) -> Tuple[jnp.ndarray, Dict[str, Any]]:
        return self.vector_field(state, control)

    # @nn.compact
    # def init_method(
    #     self,
    #     state: jnp.ndarray,
    #     control: jnp.ndarray
    # ) -> Tuple[jnp.ndarray, Dict[str, Any]]:
    #     """
    #     Initialize the engine model.
    #     """
    #     latent_init = self.calculate_initial_guess_for_latent(state)
    #     full_state = self.combine_state_and_latent(state, latent_init)
    #     vf, extra = self.vector_field(full_state, control)
    #     return vf, extra, latent_init

    def constrain_prdictions(
        self,
        state: jnp.ndarray,
    ) -> jnp.ndarray:
        """
        Apply constraints to the prediction.
        """
        state_min = jnp.array([self.state_max_min[name]["min"] for name in self.name_states])
        state_max = jnp.array([self.state_max_min[name]["max"] for name in self.name_states])
        engine_speed, boost_pressure, rear_wheelspeed = state
        engine_speed_min, boost_pressure_min, rear_wheelspeed_min = state_min
        engine_speed_max, boost_pressure_max, rear_wheelspeed_max = state_max
        return jnp.array([
            jnp.clip(engine_speed, engine_speed_min, engine_speed_max),
            jnp.clip(boost_pressure, boost_pressure_min, boost_pressure_max),
            jnp.clip(rear_wheelspeed, rear_wheelspeed_min, rear_wheelspeed_max)
        ])

In [ ]:
class ImprovedEngineDynamics(EngineDynamics):
    """
    Prototype v1 for the engine dynamics module.
    """
    stop_gradient: bool = True
    
    def with_stop_gradient(self, x):
        if self.stop_gradient:
            return jax.lax.stop_gradient(x)
        return x

    @property
    def history_size(
        self,
    ):
        """
        Get the history size.
        """
        return 10

    def setup_boost(self):
        self._boost_dot = self.create_networks(self.boost_config, 1)
        self.boost_dot = lambda x: self._boost_dot(self.with_stop_gradient(x))
        
    def setup_net_torque(self):
        self._engine_torque = self.create_networks(self.net_torque_config["engine"], 1)
        self.engine_torque = lambda x: self._engine_torque(self.with_stop_gradient(x))
        # self._shaft_torque = self.create_networks(self.net_torque_config["shaft"], 2)
        # self.shaft_torque = lambda x: self._shaft_torque(self.with_stop_gradient(x))
        # self._res_wheelspeed = self.create_networks(self.net_torque_config["res_wheelspeed"], 2)
        # self.res_wheelspeed = lambda x: self._res_wheelspeed(self.with_stop_gradient(x))

    def setup_clutch_gear_ratio(self):
        # self._clutch_gear_ratio_fn = self.create_networks(self.gear_ratio_config, 1)
        # self.clutch_gear_ratio_fn = lambda x: self._clutch_gear_ratio_fn(self.with_stop_gradient(x))
        self.clutch_max = self.get_parameter_by_name("clutch_max")

    def setup_latent(self):
        self._latent_state_net = self.create_networks(self.latent_config["gear_dynamic"], 1)
        self.latent_state_net = lambda x: self._latent_state_net(self.with_stop_gradient(x))
        # self.latent_state_net = self._latent_state_net

    def calculate_shaft_engine_torque(self, data: Dict[str, jnp.ndarray]) -> jnp.ndarray:
        # Start with the braking torque
        w_e, w_r = data["engine_speed_unscaled"], data["rear_wheelspeed_unscaled"]
        latent_space = data["latent_state"]
        curr_gear_ratio, angle_e, angle_w = latent_space
        # Calculate the delta for torsional effect on the shaft
        delta_angle = curr_gear_ratio * angle_e - angle_w
        delta_speed = curr_gear_ratio * w_e - w_r
        # shaft_in = jnp.array([delta_angle, delta_speed])
        # coeff_1, coeff_2 = jnp.exp(self.shaft_torque(shaft_in))
        coeff_1, coeff_2 = self.get_parameter_by_name("c1"), self.get_parameter_by_name("c2")
        T_s = coeff_1 * delta_angle + coeff_2 * delta_speed
        return T_s
    
    def calculate_net_torque(self, data: Dict[str, jnp.ndarray]) -> jnp.ndarray:
        # Start with the braking torque
        w_e, p_b = data["engine_speed_unscaled"], data["boost_pressure_unscaled"]
        theta = data["throttle_unscaled"]

        # Engine torque
        engine_in = jnp.array([w_e, theta, p_b])
        T_e = self.engine_torque(engine_in)[0]

        # Calculate shaft torque
        T_s = self.calculate_shaft_engine_torque(data)
        total_torque = T_e - T_s

        # Get the wheelspeed residual
        w_r = data["rear_wheelspeed_unscaled"]
        w_r_2 = w_r ** 2
        # wr_c1, wr_c2 = jnp.exp(self.res_wheelspeed(jnp.array([w_r, w_r_2])))
        wr_c1, wr_c2 = self.get_parameter_by_name("wr_c1"), self.get_parameter_by_name("wr_c2")
        wr_c3 = self.get_parameter_by_name("wr_c3") * 0.001
        wr_res = - (wr_c1 * w_r + wr_c2 * w_r_2 + wr_c3) * 0.01
        # wr_res = 0
        return total_torque, {"torque_net": total_torque, "wr_res": wr_res, "torque_shaft": T_s}

    def calculate_boost_field(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        w_e, p_b, theta = data["engine_speed"], data["boost_pressure"], data["throttle_unscaled"]
        # More side information here
        p_b_dot = self.boost_dot(jnp.array([w_e, theta, p_b]))[0]
        return p_b_dot, {}

    def calculate_engine_speed_field(
        self,
        data: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        # Calculate the engine speed derivative
        T_net, _extra = self.calculate_net_torque(data)
        w_e_dot = T_net # * self.get_parameter_by_name("inv_J_e")
        return w_e_dot, _extra

    def calculate_clutch_gear_ratio(self, data):
        w_e = data["engine_speed_unscaled"]
        w_r = data["rear_wheelspeed_unscaled"]
        clucth_val =  w_r / w_e
        # Let's extract the shaft torque
        # clucth_val_nn  = self.clutch_gear_ratio_fn(jnp.array([data["engine_speed"],data["rear_wheelspeed"]]))[0]
        # clucth_val = jnp.exp(clucth_val_nn) * clucth_val
        return clucth_val, {"clutch" : 1} # Inverse of the clutch

    def calculate_initial_guess_for_latent(
        self,
        state : jnp.ndarray,
        control: jnp.ndarray,
        times: jnp.ndarray
    ) -> jnp.ndarray:
        if self.is_initializing():
            # Repeat state and control the adequate number of times
            state =  state[None]
            theta_e = 0
            theta_w = 0
        else:
            # Extract sequence of both engine speed and wheel speed
            engine_speed = state[:, 0]
            wheel_speed = state[:, 2]
            dt_time = jnp.diff(times)
            # Integrate them to get the angles
            theta_e = jnp.sum(engine_speed[:-1] * dt_time)
            theta_w = jnp.sum(wheel_speed[:-1] * dt_time)
        # Normalize the state and control
        feats = self.get_data_from_state_and_control(state[-1], jnp.zeros(self.num_controls))
        gear_ratio_init = self.calculate_clutch_gear_ratio(feats)[0]
        # print(f"Initial guess: {gear_ratio_init}, {theta_e}, {theta_w}")
        # Calculate the initial guess
        return jnp.array([gear_ratio_init, theta_e, theta_w])

    def calculate_latent_state_field(
        self,
        data: Dict[str, jnp.ndarray],
        extra: Dict[str, jnp.ndarray]
    ) -> Tuple[jnp.ndarray, Dict[str, jnp.ndarray]]:
        # Calculate the latent state derivative
        latent_state = data["latent_state"]
        # Derivative of the angles
        engine_speed_u = data["engine_speed_unscaled"]
        rear_wheelspeed_u = data["rear_wheelspeed_unscaled"]
        # Dynamics of the gear ratio depends on the engine speed and rear wheelspeed, torque, and current gear ratio
        engine_speed = data["engine_speed"]
        # rear_wheelspeed = data["rear_wheelspeed"]
        gear_ratio = latent_state[0]
        torque_shaft = extra["torque_shaft"]
        # gear_ratio_dot = self.latent_state_net(jnp.array([engine_speed, torque_shaft, gear_ratio]))[0]
        gear_ratio_eq  = self.latent_state_net(jnp.array([engine_speed, torque_shaft]))[0]
        gear_ratio_dot = (gear_ratio_eq - gear_ratio) * self.get_parameter_by_name("tau_gear")
        return jnp.array([gear_ratio_dot, engine_speed_u, rear_wheelspeed_u]), {}

    def vector_field(
        self,
        state: jnp.ndarray,
        control: jnp.ndarray
    ) -> Tuple[jnp.ndarray, Dict[str, Any]]:
        """
        Combine the individual state derivative calculations to produce the full drift vector.
        Also aggregates auxiliary outputs in an extras dictionary.
        """
        # Check if the module is being initialized.
        if self.is_initializing():
            # Get initial latent state
            latent_init = self.calculate_initial_guess_for_latent(state, control, None)
            state =  self.combine_state_and_latent(state, latent_init)

        # First, extract the latent state if it exists.
        state, latent_state = self.split_state_latent_var(state)

        # The data
        data = self.get_data_from_state_and_control(state, control)
        # Add the latent space to data
        if latent_state is not None:
            data = {**data, "latent_state": latent_state}

        dot_omega_e, extras_speed  = self.calculate_engine_speed_field(data)
        dot_p_b,     extras_boost  = self.calculate_boost_field(data)
        inv_gear_ratio, _angle_e, _angle_w = latent_state
        
        # Let's extrapolate rear_wheel speed evolution.
        moment_coeffs = self.get_parameter_by_name("JeJr_1")
        torque_shaft = extras_speed["torque_shaft"]
        gear_ratio = 1 / (inv_gear_ratio + 1.0e-3)
        dot_wr = gear_ratio * torque_shaft * moment_coeffs + extras_speed.get("wr_res", 0)
        
        # Assemble drift. For example, add any clutch correction to the gear ratio derivative.
        drift = jnp.array([dot_omega_e, dot_p_b, dot_wr]) # * 100
        
        # Let care about the latent state.
        latent_state_dot, extra_latent = self.calculate_latent_state_field(data, extras_speed)
        drift = self.combine_state_and_latent(drift, latent_state_dot)
        
        # Merge all extras.
        extras = {
            **extras_speed, 
            **extras_boost,
            **extra_latent,
            "clutch": jnp.square(self.clutch_max),
        }
        return drift, extras

    def constrain_latent(
        self,
        state: jnp.ndarray,
    ) -> jnp.ndarray:
        """
        Apply constraints to the prediction.
        """
        # Lower bound is zero for all latent variables
        return jnp.clip(state, 0, None)
        

In [ ]:
def integrate_path(
    init_state: jnp.ndarray,
    controls: jnp.ndarray,
    times: jnp.ndarray,
    rng: jax.random.PRNGKey,
    params: Dict[str, Any], # Parameters for the dynamics model and noise
    history_data: Any,
    dynamics: EngineDynamics,
    noise: NNNoiseTerm,
    return_full_state: bool = False,
    num_iter_in: int = 1,
) -> jnp.ndarray:
    """
    Integrate the dynamics using the Euler–Maruyama method.

    The next state is assumed to be distributed as
      x_{k+1} ~ N(x_k + f(x_k, u_k) * dt,  (sqrt(dt)*noise)^2 )
    where 'f(x_k, u_k)' is given by the dynamics model and 'noise'
    is either provided by noise_fn or sampled from a standard normal distribution.

    Args:
        init_state: (n,) array representing the initial state.
        controls: (T, m) array of controls (one per time step).
        times: (T+1,) array of time points. (dt = times[i+1] - times[i])
        rng: a JAX random key.
        params: a dictionary of parameters for the dynamics model and noise.
        dynamics: an instance of EngineDynamics.
        noise: an instance of BasicNoiseTerm

    Returns:
        path: (T+1, n) array of integrated states.
        extras: a dictionary of auxiliary outputs.
    """
    # Compute time increments: dt for each integration step.
    dt_array = jnp.diff(times)  # shape: (T,)

    # num_iter_in subdivise each dt into smaller steps dt/num_iter_in
    dt_array = jnp.repeat(dt_array / num_iter_in, num_iter_in)
    # Do the same for the controls
    controls = jnp.repeat(controls, num_iter_in, axis=0)

    # Let's create the variables for the normal noise.
    z_norm = jax.random.normal(rng, shape=(dt_array.shape[0], init_state.shape[0]))
    
    # Define the step function for scanning.
    def step_fn(carry, inputs):
        state = carry
        control, dt, z = inputs  # dt is a scalar for this step

        # Compute the drift using the dynamics model.
        drift, _extra = dynamics.apply(params["drift"], state, control)
        mean_update = state + drift * dt
        mean_update_reduced, mean_update_latent = dynamics.split_state_latent_var(mean_update)
        # _extra["mean_update"] = dynamics.constrain_prdictions(mean_update)

        # Compute the noise term.
        reduced_state, _ = dynamics.split_state_latent_var(state)
        noise_term, noise_extra = noise.apply(params["noise"], reduced_state, control)
        noise_sqrt_dt = noise_term * jnp.sqrt(dt)
        
        # Euler–Maruyama update: add noise scaled by sqrt(dt).
        next_state = mean_update_reduced + noise_sqrt_dt * z
        _extra = {
            **_extra,
            **noise_extra,
            "noise_sqrt": noise_sqrt_dt,
        }
        # Let's clip the state
        next_state = dynamics.constrain_prdictions(next_state)
        mean_update_latent = dynamics.constrain_latent(mean_update_latent)
        full_next_state = dynamics.combine_state_and_latent(next_state, mean_update_latent)
        if return_full_state:
            return full_next_state, (full_next_state, _extra)
        return full_next_state, (next_state, _extra)

    # Let's start with obtaining the latent state information. let's call apply with the name of function
    latent_initial_state = dynamics.apply(
        params["drift"], *history_data, method="calculate_initial_guess_for_latent"
    )
    # latent_initial_state = dynamics.calculate_initial_guess_for_latent(init_state)
    full_state = dynamics.combine_state_and_latent(init_state, latent_initial_state)
    # Use scan to integrate the dynamics.
    _, (path, extras) = jax.lax.scan(
        step_fn, full_state, (controls, dt_array, z_norm)
    )
    
    # Because we subdivide the time steps, we need to return only every num_iter_in steps
    extras = jax.tree_map(lambda x: x[::num_iter_in], extras)
    
    if return_full_state:
        return jnp.concatenate([full_state[None], path], axis=0)[::num_iter_in], extras

    return jnp.concatenate([init_state[None], path], axis=0)[::num_iter_in], extras

In [ ]:
def compute_nll_loss(
    params: Dict[str, Any],
    states: jnp.ndarray,     # shape (T+1, n)
    controls: jnp.ndarray,   # shape (T, m)
    times: jnp.ndarray,      # shape (T+1,)
    rng: jax.random.PRNGKey,
    history_data: Any,
    dynamics: EngineDynamics,
    noise: NNNoiseTerm,
    discounts: jnp.ndarray,
    num_iter_in: int = 1,
    scaling_factor: jnp.ndarray = 1.0
):
    # Check if rng is a single key
    if rng.ndim == 1:
        rng = rng[None]

    # Integrate the dynamics - > We just do one path for now
    path, extras = jax.vmap(
        integrate_path, in_axes=(None, None, None, 0, None, None, None, None, None, None)
    )(states[0], controls, times, rng, params, history_data, dynamics, noise, False, num_iter_in)
    # path, extras = integrate_path(
    #     states[0], controls, times, rng, params, dynamics, noise
    # )
    # Extract noise term
    noise_term = extras["noise_sqrt"]
    if noise.ignore_noise:
        noise_term = jnp.ones_like(noise_term)
    # Compute the log-likelihood of the path.
    diff_state_no_scale = (path[:,1:,:] - states[1:,:][None])
    diff_state = (diff_state_no_scale / noise_term) * scaling_factor
    noise_loss = jnp.mean(jnp.sum( jnp.sum( jnp.log(noise_term), axis=-1 ) * discounts[None], axis=-1 ))
    log_likelihood = jnp.mean(jnp.sum( jnp.sum( diff_state ** 2, axis=-1 ) * discounts[None], axis=-1 ))
    mse = jnp.mean(jnp.mean( (diff_state_no_scale ** 2) * discounts.reshape(1, -1, 1), axis=1 ), axis=0)
    ret_dict = {
        "nll_noise" : noise_loss,
        "nll_mse" : log_likelihood,
        "mse": jnp.sum(mse)
    }
    return ret_dict, (mse, path), extras

In [ ]:
# Add these mappings at the top of the file (or in an appropriate common module)
DYNAMICS_CLASSES = {
    "EngineDynamics": EngineDynamics,
    "ImprovedEngineDynamics": ImprovedEngineDynamics,
    # Add other dynamics classes here if needed.
}

NOISE_CLASSES = {
    "NNNoiseTerm": NNNoiseTerm,
    # Add other noise classes here if needed.
}

def load_model_from_config(
    model_config: Dict[str, Any],
    seed: int = 0,
    mean_data_var: Dict[str, Any] = {},
    scale_data_var: Dict[str, Any] = {},
):
    # Instantiate Drift / dynamics model from mapping.
    drift_conf = model_config["drift"]
    drift_type = drift_conf["type"]
    drift_args = drift_conf.get("args", {})
    if "state_action_mean" not in drift_args:
        drift_args["state_action_mean"] = mean_data_var
    if "state_action_std" not in drift_args:
        drift_args["state_action_std"] = scale_data_var

    if drift_type not in DYNAMICS_CLASSES:
        raise ValueError(
            f"Drift type {drift_type} not recognized. \nAvailable types: {list(DYNAMICS_CLASSES.keys())}"
        )
    drift = DYNAMICS_CLASSES[drift_type](**drift_args)
    
    # Instantiate Noise module from mapping.
    noise_conf = model_config["noise"]
    noise_type = noise_conf["type"]
    noise_args = noise_conf.get("args", {})
    noise_args["output_dimension"] = drift.num_states

    if noise_type not in NOISE_CLASSES:
        raise ValueError(
            f"Noise type {noise_type} not recognized. \nAvailable types: {list(NOISE_CLASSES.keys())}"
        )
    
    # Set get_normalized_state as an attribute of NOISE_CLASSES[noise_type]
    noise_class = NOISE_CLASSES[noise_type]
    noise_class.get_normalized_state = drift.get_normalized_state
    noise = noise_class(**noise_args)
    
    # Set up PRNG keys using the provided seed.
    rng = jax.random.PRNGKey(seed)
    rng, rng_drift, rng_noise = jax.random.split(rng, 3)
    
    # Use dynamic model properties to create dummy inputs for initialization.
    # EngineDynamics.__call__ is assumed to take (state, control) as arguments.
    dummy_state = jnp.zeros((drift.num_states,))
    dummy_control = jnp.zeros((drift.num_controls,))
    drift_params = drift.init(rng_drift, dummy_state, dummy_control)
    # res = drift.apply(drift_params, jnp.ones((5,)), dummy_control)
    
    # For the noise module, assume its __call__ takes an input with the same dimension as the state.
    dummy_noise_input = jnp.zeros((drift.num_states,))
    noise_params = noise.init(rng_noise, dummy_noise_input, dummy_control)

    # Merge the parameters into a single dictionary.
    params = {"drift": drift_params, "noise": noise_params}
    
    return drift, noise, params

In [ ]:
##### Set of functions for formatting the dataset in a structured way for training ####
def find_consecutive_true(metrics, min_length=-1):
    """ Given a boolean array, find the consecutive sections 
        of True values that are at least min_length long.
    
    Args:
        metrics: The boolean array
            (n,) array
        min_length: The minimum length of the consecutive 
        True sections
            int
    
    Returns:
        full_auto_inds: The indices of the consecutive 
        True sections
            list of arrays
    """
    full_auto = metrics
    section_inds_ = np.split(
        np.r_[:len(full_auto)], 
        np.where(np.diff(full_auto) != 0)[0]+1
    )
    full_auto_inds = []
    for inds_ in section_inds_:
        if full_auto[inds_[0]] and len(inds_) > min_length:
            full_auto_inds.append(inds_)
    return full_auto_inds

def format_gtstate_traj(
    metadata: Dict[str, Any],
    data: Dict[str, np.ndarray],
    valid_trajectory_cond: callable = None,
    min_traj_len: int = 200,
    indx_traj: int = 0
):
    """
    Format the ground truth states and controls in a format compatible with dutils.
    
    Args:
        data: dictionary containing the ground truth states and controls
            dict
        valid_trajectory_cond: condition to filter out invalid trajectories or transitions
            callable
    """
    if valid_trajectory_cond is not None:
        valid = valid_trajectory_cond(data)
    else:
        valid = np.ones(data["time"].shape, dtype=bool)
    valid_indexes = find_consecutive_true(valid, min_length=min_traj_len)
    # If no valid section is found, skip the run
    success = True
    if len(valid_indexes) == 0:
        success = False
        tqdm.write(
            f'No subset of length < {min_traj_len} is invertible'
        )
        tqdm.write('-----------------\n')
        return (None, None), success
    # Extract the valid transitions
    trajectories = []
    trajectories_info = []
    for seq in valid_indexes:
        # Extract the valid transitions
        curr_data = {}
        max_min_data = {}
        for key, value in data.items():
            curr_data[key] = value[seq]
            max_min_data[key] = (np.max(curr_data[key]), np.min(curr_data[key]))
        trajectories.append(curr_data)
        curr_traj_info = {
            **metadata,
            'duration (s)' : curr_data['time'][-1] - curr_data['time'][0],
            'traj_idx' : indx_traj + len(trajectories) - 1,
            'max_min_data' : max_min_data,
            "system_physical_params": {}, # TODO: Completed later
        }
        trajectories_info.append(curr_traj_info)
    return (trajectories, trajectories_info), success

def exponential_moving_average(data, alpha=0.2):
    """
    Compute the exponential moving average of a 1D array.

    Args:
        data (np.ndarray): Input data array.
        alpha (float): Smoothing factor between 0 and 1. Higher alpha means less smoothing.
    
    Returns:
        np.ndarray: Filtered data array.
    """
    ema = np.zeros_like(data)
    ema[0] = data[0]
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1 - alpha) * ema[i - 1]
    return ema

def preprocess_data(
    data: Dict[str, np.ndarray],
    alpha: float = 0.2,
    torque_scaling: float = 1.0,
    scale_wr: float = 0.1,
    scale_pb: float = 0.1,
):
    """
    Perform the necessary processing of the data.
    Typically renaming some convenient states and controls.
    Converting to the right units.
    etc...
    """
    alpha_filtering = {
        "rear_wheelspeed": 0.2,
        "engine_speed": 0.8,
        "boost_pressure": 0.2,
    }
    # Average rear wheel speed into a single variable
    w_rl = data["wheelRR_angVel"] # In radian per second
    w_rr = data["wheelRL_angVel"] # In radian per second
    data["rear_wheelspeed"] = 0.5 * (w_rl + w_rr) * scale_wr
    # Convert RPM into radian per second
    data["engine_speed"] = data["RPM"] * (2 * np.pi / 60) * scale_wr
    # Rename boost pressure
    data["boost_pressure"] = data["turboBoost"] * scale_pb
    # Let's apply simple filtering for the accelerometer on x value only for now
    data["accel_x_filtered"] = exponential_moving_average(data["accel_x"], alpha = alpha)
    # Let's estimate the torque by using the pure rolling assumption
    # In this case a = R wr_dot = R (1/Inertia) (wheel_torque)
    # Thus, wheel_torque ~ J a  / R, torque_scaling = J/R
    data["wheel_torque_est"] = data["accel_x_filtered"] * torque_scaling
    data["throttle_old"] = data["throttle"]
    data["throttle"] = data.get("throttleValve", data["throttle"])
    # Filter the necessary states
    for key, alpha_val in alpha_filtering.items():
        data[key] = exponential_moving_average(data[key], alpha=alpha_val)
    # Add more here for completion

def format_dataset(
    runs: List[int],
    sensor_type: str = "gtstate",
    vehicle_model: str = "utv_wild",
    alpha: float = 0.2,
    torque_scaling: float = 1.0,
    valid_trajectory_cond: callable = None,
    min_traj_len: int = 200,
):
    """
    Main function to format the runs in a more structured way for training the engine model.
    """
    # The main variables
    m_trajectories = []
    m_trajectories_info = []
    # Loading logging
    failed_logs = []
    successful_logs = []
    for run in runs:
        run_name = f"Run {run}"
        tqdm.write(f'\n========== Loading {run_name} ==========\n')
        # Load the run first
        try:
            metadata, data = load_log_data(run)
            metadata.pop('sim', None)
        except Exception:
            traceback_str = traceback.format_exc()
            tqdm.write(traceback_str)
            failed_logs.append(run_name)
            tqdm.write('Error loading...')
            tqdm.write('-----------------\n')
            continue
        # Load was successful - Let's iterate through all vehicle sensor to pick relevant one
        data_of_interest = {}
        for (vehicle_name, veh_sensor), curr_data in data.items():
            veh_model_type = metadata[vehicle_name]
            if veh_sensor != sensor_type or veh_model_type != vehicle_model:
                tqdm.write(f"Ignoring {vehicle_name} - {veh_sensor} of type {veh_model_type} from {run_name}")
                continue
            # Otherwise, this is a valid data
            data_of_interest[(vehicle_name, veh_sensor)] = curr_data 
        if len(data_of_interest) < 1:
            tqdm.write(f"No useful data from {run_name}. Skipping.....")
            failed_logs.append(run_name)
            continue
        # Otherwise, we have useful data.
        num_success = 0
        for info, curr_data in data_of_interest.items():
            tqdm.write(f"Processing {info}....")
            curr_data =  {k : np.array(v) for k, v in curr_data.items()}
            # Let's process the data
            preprocess_data(curr_data, alpha = alpha, torque_scaling = torque_scaling)
            # Now let's try to split it.
            (_trajectories, _trajectories_info), _success = format_gtstate_traj(
                metadata, curr_data, valid_trajectory_cond = valid_trajectory_cond,
                min_traj_len=min_traj_len, indx_traj=len(m_trajectories)
            )
            if not _success:
                continue
            num_success += 1
            # Trajectories were loaded successfully, so let's append them
            m_trajectories.extend(_trajectories)
            m_trajectories_info.extend(_trajectories_info)
        if num_success >= 1:
            successful_logs.append(run_name)
        else:
            failed_logs.append(run_name)
    # Let's print out a summary of successful and failed loading
    print(f"Total number of successful logs: {len(successful_logs)}")
    print(f"Total number of failed logs: {len(failed_logs)}")
    print(f"Total number of trajectories: {len(m_trajectories)}")

    # Print failed loading
    if len(failed_logs) > 0:
        print('\nFailed loading : ')
        for log_info in failed_logs:
            print('- ', log_info)

    if len(m_trajectories) == 0:
        print("No valid trajectory found. Exiting...")
        return

    if len(successful_logs) > 0:
        print('\nSuccessful loading : ')
        for log_info in successful_logs:
            print('- ', log_info)

    # Dataset
    dataset ={
        'trajectories' : m_trajectories,
        'trajectories_info' : m_trajectories_info,
        'system_physical_params' : {}, # To be updated
        'data_fields': list(m_trajectories[0].keys()),
    }
    return dataset

def get_max_min_data(
    dataset: Dict[str, Any],
    fields: List[str]
):
    """
    Get the maximum and minimum values of the fields in the dataset.
    """
    max_min_data = {}
    for field in fields:
        max_min_data[field] = {
            "max" : -np.inf,
            "min" : np.inf
        }
    for traj in dataset["trajectories"]:
        for field in fields:
            max_min_data[field]["max"] = max(max_min_data[field]["max"], np.max(traj[field])+1)
            max_min_data[field]["min"] = min(max_min_data[field]["min"], np.min(traj[field])-1)
    return max_min_data

In [ ]:
# Let's build the loss function and set that up
def map_gradient_update_params(
    params : Dict[str, Any],
    opt_state : Any,
    state : jnp.ndarray,
    control : jnp.ndarray,
    extra_variables : Dict[str, Any],
    rng_key : jnp.ndarray,
    loss_fn : Callable,
    opt : optax.GradientTransformation,
) -> Tuple[Dict[str, Any], Dict[str, Any], Dict[str, jnp.ndarray]]:
    """
    Compute the gradient and update the parameters of the neural SDE model.
    
    Args:
        params: The parameters of the neural SDE model
            dict
        opt_state: The state of the optimizer
            Any
        state: The state of the system
            (n,) array
        control: The control of the system
            (m,) array
        extra_variables: The time dependent parameters
            dict
        rng_key: The random number generator key
            (2,) array
        loss_fn: The loss function
            callable
        opt: The optimizer
            optax.GradientTransformation
    
    Returns:
        Dict[str, Any]: The updated parameters
        Dict[str, Any]: The updated optimizer state
        Dict[str, jnp.ndarray]: The feature values from the loss function
    """
    # Split the key
    rng_key = jax.random.split(rng_key, state.shape[0])
    # By default only differentiate with respect to params
    grads, featvals = jax.grad(loss_fn, has_aux=True)(
        params, state, control, extra_variables, rng_key
    )
    updates, opt_state = opt.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, featvals


def single_sequence_training_loss(
    params : Dict[str, Any],
    state : jnp.ndarray,
    control : jnp.ndarray,
    extra_variables : Dict[str, Any],
    rng_key : jnp.ndarray,
    extra_loss_args : Dict[str, Any],
    drift_term : EngineDynamics,
    noise_term : NNNoiseTerm
):
    """
    Implement the training loss function for a single sequence.
    """
    assert state.shape[0] >= drift_term.history_size + 1, \
        "The state should have at least the history size + 1"
    time_val = state[..., -1]
    state  = state[..., :-1]
    # Lets split state and control to account for the history
    (hist_state, hist_control), (state, control) = \
        drift_term.split_state_with_history(state, control)
    t_hist = time_val[:hist_state.shape[0]]
    hist = (hist_state, hist_control, t_hist)
    time_val = time_val[-state.shape[0]:]
    
    num_samples = extra_loss_args.get("num_samples", 1)
    # Create the discounts array
    if "discount" not in extra_loss_args:
        discounts = jnp.ones_like(control)
    else:
        discount = extra_loss_args["discount"]
        dt_scaling =  extra_loss_args.get("discount_dt", 0.01)
        discounts = discount ** ( (time_val - time_val[0])[:-1] / dt_scaling)
    
    if num_samples > 1:
        rng_key = jax.random.split(rng_key, num_samples)

    # Let's calculate the loss
    loss_dict, (mse_state, pred_state), extras = compute_nll_loss(
        params, state, control, time_val, rng_key, hist, drift_term,
        noise_term, discounts, extra_loss_args.get("num_inter_in", 1),
        extra_loss_args.get("scaling_factor", 1)
    )
    # Now extract the wheelspeed from the extra variables
    clutch_val  = jnp.mean(extras["clutch"], axis=0)
    state_loss = { k : _err for k, _err in zip(drift_term.name_states, mse_state)}
    # Get noise max value
    noise_max = jnp.sum(extras["max_noise"][0,0])
    # All loss
    losses  = {
        **loss_dict,
        **state_loss,
        "loss_clutch": jnp.sum(jnp.square(clutch_val - 1) * discounts),
        "loss_noise_max": noise_max
    }
    return losses

# Let's define the batch version of the training loss
def batch_training_loss(
    params : Dict[str, Any],
    state : jnp.ndarray,
    control : jnp.ndarray,
    extra_variables : Dict[str, Any],
    rng_key : jnp.ndarray,
    extra_loss_args : Dict[str, Any],
    drift_term : EngineDynamics,
    noise_term : NNNoiseTerm
):
    if len(rng_key.shape) == 1:
        rng_key = jax.random.split(rng_key, state.shape[0])

    loss_vals = jax.vmap(
        single_sequence_training_loss,in_axes=(None, 0, 0, 0, 0, None, None, None)
    )(params, state, control, extra_variables, rng_key, extra_loss_args["pred"], drift_term, noise_term)
    losses = {k: jnp.mean(v) for k, v in loss_vals.items()}
    # Regularization loss
    reg_loss = jnp.sum( jnp.array([jnp.sum(jnp.square(v)) for v in jax.tree_leaves(params)]) )
    losses = {**losses, "loss_reg": reg_loss}
    total_loss = jnp.array([losses[k] * reg_v for k, reg_v in extra_loss_args["reg_params"].items()]).sum()
    return total_loss, {**losses, "total_loss": total_loss}


In [ ]:
# Now let's start setting up the training optimizer and such
def build_optimizer(
    opt_config : Dict[str, Any]
) -> optax.GradientTransformation:
    """ Build the optimizer from a configuration dictionary
    
    Args:
        opt_config: The configuration of the optimizer
            dict
    
    Returns:
        optax.GradientTransformation: The optimizer
    """
    chain_list = []
    for elem in opt_config:
        name_elem = elem['name']
        m_fn = getattr(optax, name_elem)
        m_params = elem.get('params', {})
        print(f'Function : {name_elem} | params : {m_params}')
        if elem.get('scheduler', False):
            m_params = m_fn(**m_params)
            chain_list.append(optax.scale_by_schedule(m_params))
        else:
            chain_list.append(m_fn(**m_params))
    # Return the optimizer
    return optax.chain(*chain_list)

In [ ]:
# Utility functions for data sampling and logging training progress
from logging_utils import TrainCheckpoints
import data_utils as dutils
from data_utils import (
    split_dataset,
    pick_batch_transitions_as_array,
    remove_irrelevant_fields,
    split_time_dependent_parameters,
    get_mean_and_scale_fields,
    sequential_loader_full_dataset
)
TRAINING_LOGS_DIR = os.getcwd() + "/training_files/"


In [ ]:
# Let;s load the data that will be used to train the models
def criteria_valid_traj(
    data : Dict[str, np.array]
):
    # First the vehicle must be in realistic drive mode (as the mode for control)
    is_realistic = data["driveStatus_isRealisticDrive"] == 1
    # For now let's assume 2WD and ig range mode (0)
    valid = (data["brake"] <= 0.001) & \
            (data["pbrake"] <= 0.0001) & \
            (data["driveStatus_mode4WD"] == 0) & \
            (data["driveStatus_modeRangeBox"] == 0) & \
            (data["rear_wheelspeed"] >= 0.5) & \
            (data["driveStatus_engineRunning"] == 1) & \
            is_realistic
    # valid  = valid & (np.logical_and(data["time"] >141, data["time"] < 155))
    return valid

# Let's test all of that
m_runs =[9,]
full_dataset = format_dataset(
    m_runs,
    sensor_type = "gtstate",
    vehicle_model = "utv_wild",
    alpha = 0.05,
    torque_scaling = 1.0, # No meaning for now
    valid_trajectory_cond = criteria_valid_traj,
    min_traj_len = 200,
)

In [ ]:
# List some of the trajectories infos
for traj_info in full_dataset["trajectories_info"]:
    print({k : v for k, v in traj_info.items() if k != "max_min_data" and k != "system_physical_params"})

In [ ]:
full_dataset["data_fields"]

In [ ]:
# Let's do some data splitting
_no_split = True
_test_ratio = 0.1
_seed = 42
np.random.seed(_seed)


useful_fields = ["engine_speed", "boost_pressure", "throttle", "rear_wheelspeed", "time"]
# Remove the irrelevant fields from the dataset
_full_dataset = remove_irrelevant_fields(
    full_dataset, useful_fields
)
# Compute the mean and scale of the relevant fields
# for input normalization of the neural networks
mean_and_scale = get_mean_and_scale_fields(_full_dataset)
if _no_split:
    test_data, train_data = _full_dataset, _full_dataset
else:
    train_data, test_data = split_dataset(_full_dataset, _test_ratio, _seed)

In [ ]:
# Define the model and initialize its parameters
model_config = {
    # "drift": {
    #     "type": "EngineDynamicsV1",
    #     "args": {
    #         "net_torque_config": {
    #             "throttle_coef": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (128, 128),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "throttle": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "braking": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.1
    #                 }
    #             }
    #         },
    #         "boost_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (128, 128),
    #                 "activation_fn": "relu",
    #                 "initial_value_range": 0.1
    #             }
    #         },
    #         "gear_ratio_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (32, 32),
    #                 "activation_fn": "relu",
    #                 "initial_value_range": 0.01
    #             }
    #         },
    #         "guided_parameters": {
    #             "inv_J_e": 0.1
    #         },
    #         "use_normalized_input": True
    #     }
    # },
    
    # "drift": {
    #     "type": "EngineDynamicsV3",
    #     "args": {
    #         "net_torque_config": {
    #             "throttle": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64, 32),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "braking": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (32, 32),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "res_wheelspeed": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 0.1
    #                 }
    #             }
    #         },
    #         "boost_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (64, 64, 32),
    #                 "activation_fn": "tanh",
    #                 "initial_value_range": 0.1
    #             }
    #         },
    #         "gear_ratio_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (32, 32),
    #                 "activation_fn": "tanh",
    #                 "initial_value_range": 0.01
    #             }
    #         },
    #         "guided_parameters": {
    #             "inv_J_e": 0.1
    #         },
    #         "use_normalized_input": True
    #     }
    # },
    "drift":{
        "type": "ImprovedEngineDynamics",
        "args": {
            "net_torque_config": {
                "engine": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": (128, 128),
                        "activation_fn": "tanh",
                        "initial_value_range": 0.1,
                        "fake_normalization": True
                    }
                },
                "shaft": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": (32, 32),
                        "activation_fn": "tanh",
                        "initial_value_range": 0.1,
                        "fake_normalization": True
                    }
                },
                "res_wheelspeed": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": (32, 32),
                        "activation_fn": "tanh",
                        "initial_value_range": 0.1
                    }
                },
            },
            "boost_config": {
                "type": "MLP",
                "args": {
                    "layers_archictecture": (64, 64),
                    "activation_fn": "tanh",
                    "initial_value_range": 0.1,
                    "fake_normalization": True
                }
            },
            "gear_ratio_config": {
                "type": "MLP",
                "args": {
                    "layers_archictecture": (32, 32),
                    "activation_fn": "tanh",
                    "initial_value_range": 0.01
                }
            },
            "latent_config": {
                "gear_dynamic": {
                    "type": "MLP",
                    "args": {
                        "layers_archictecture": (32, 32),
                        "activation_fn": "tanh",
                        "initial_value_range": 0.01,
                        "fake_normalization": True
                    }
                },
            },  
            "guided_parameters": {},
            "use_normalized_input": True,
            "stop_gradient": True,
        }
    },
    
    # "drift": {
    #     "type": "EngineDynamicsLatent",
    #     "args": {
    #         "net_torque_config": {
    #             "throttle": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "braking": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "res_wheelspeed": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (32, 32),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.1
    #                 }
    #             }
    #         },
    #         "boost_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (64, 64),
    #                 "activation_fn": "relu",
    #                 "initial_value_range": 0.1
    #             }
    #         },
    #         "gear_ratio_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (32, 32),
    #                 "activation_fn": "relu",
    #                 "initial_value_range": 0.01
    #             }
    #         },
    #         "latent_config": {
    #             "initial_guess": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (32, 32),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.01,
    #                     "output_dimension": 1
    #                 }
    #             },
    #             "vectorfield": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (32, 32),
    #                     "activation_fn": "relu",
    #                     "initial_value_range": 0.01,
    #                     "output_dimension": 1
    #                 }
    #             }
    #         },
    #         "guided_parameters": {
    #             "inv_J_e": 0.1
    #         },
    #         "use_normalized_input": True
    #     }
    # },

    # "drift": {
    #     "type": "EngineDynamicsLatentV2",
    #     "args": {
    #         "net_torque_config": {
    #             "throttle": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 0.1
    #                 }
    #             },
    #             "res_wheelspeed": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (32, 32),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 0.1
    #                 }
    #             }
    #         },
    #         "boost_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (64, 64),
    #                 "activation_fn": "tanh",
    #                 "initial_value_range": 0.1
    #             }
    #         },
    #         "gear_ratio_config": {
    #             "type": "MLP",
    #             "args": {
    #                 "layers_archictecture": (32, 32),
    #                 "activation_fn": "tanh",
    #                 "initial_value_range": 0.01
    #             }
    #         },
    #         "latent_config": {
    #             "initial_guess": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 1.0,
    #                     "output_dimension": 3
    #                 }
    #             },
    #             "vectorfield": {
    #                 "type": "MLP",
    #                 "args": {
    #                     "layers_archictecture": (64, 64),
    #                     "activation_fn": "tanh",
    #                     "initial_value_range": 1,
    #                     "output_dimension": 3
    #                 }
    #             }
    #         },
    #         "guided_parameters": {
    #             "inv_J_e": 0.1
    #         },
    #         "use_normalized_input": True
    #     }
    # },
    
    # "noise": {
    #     "type": "BasicNoiseTerm",
    #     "args": {}
    # }
    "noise": {
        "type": "NNNoiseTerm",
        "args": {
            "ignore_noise": False,
            "max_noise": jnp.array([1, 0.3, 0.3]),
            "noise_network_config": {
                "type": "MLP",
                "args": {
                    "layers_archictecture": (64, 64),
                    "activation_fn": "tanh",
                    "initial_value_range": 0.1
                }
            },
        }
    }
}

# Let's get the max and min data
max_min_data = get_max_min_data(train_data, useful_fields)
model_config["drift"]["args"]["state_max_min"] = max_min_data

In [ ]:
max_min_data

In [ ]:
m_drift, m_noise, my_params = load_model_from_config(
    model_config, seed=_seed, mean_data_var=mean_and_scale[0], scale_data_var=mean_and_scale[1]
)

In [ ]:
m_drift.use_normalized_input

In [ ]:
my_params

In [ ]:

# Create the optimizer
opt_config = [
    { "name": "scale_by_adam",},
    {
        "name": "linear_schedule",
        "scheduler": True,
        "params": {
            "init_value": -0.001,
            "end_value": -0.001,
            "transition_steps": 20000
        }
    }
]
opt = build_optimizer(opt_config)
opt_state = opt.init(my_params)

# Setup some training parameters
stepsize_range =[1, 4]
horizon = 100
stepsize_range_test = [1, 4]
horizon_test = 100

# Add the history size
horizon = horizon + m_drift.history_size
horizon_test = horizon_test + m_drift.history_size

# Problem config for data extraction
problem_config_for_dataset_extraction ={
    "names_states": list(m_drift.name_states) + ["time",],
    "names_controls": list(m_drift.name_controls),
    "time_dependent_parameters": ["rear_wheelspeed",],
    "system_physical_params": {},
}
action_sampling_strategy ={
    "default": "first"
}

# # Training parameters
# train_batch_data = pick_batch_transitions_as_array(
#     train_data, 10, stepsize_range, horizon, problem_config_for_dataset_extraction, action_sampling_strategy   
# )

# Let's define the training loop
train_batch = 32 # Number of sequence of trajectories in a batch
test_batch = 32 # Number of sequence of trajectories in a batch
num_epochs = 500 # Number of epochs
num_batches = 500 # Number of batches iteration per epoch
num_test_batches = 30
_test_freq = 1 # Frequence in terms of number of batches for testing the model
_save_freq = 1 # Frequency in terms of number of batches for saving the model
do_iterate_through_all = False # Use the full batch for training

if do_iterate_through_all:
    load_batch, traj_indexes, num_batches = sequential_loader_full_dataset(
        train_data, train_batch, stepsize_range, horizon,
        problem_config_for_dataset_extraction, action_sampling_strategy
    )

test_freq = int(num_batches * _test_freq)
save_freq = int(num_batches * _save_freq)

# Set up the loss function
reg_loss_args = {
    "pred":{
      "discount": 0.99,
      "discount_dt": np.mean(np.diff(train_data["trajectories"][0]["time"])),
      "num_samples": 4,
      "num_inter_in": 1,
      "scaling_factor": 1.0 / jnp.array([90, 2.0, 10]).reshape(1,1,-1)
    },
    "reg_params": {
        "nll_mse": 1,
        "nll_noise": 1,
        "loss_clutch": 1,
        "loss_reg": 1.0e-8,
        "loss_noise_max": 0
    }
}

In [ ]:
# Create the loss function and jit it. Strip it down to the bare minimum for now
jit_model_loss = jax.jit(
    lambda params, state, control, extra_variables, rng_key : \
    batch_training_loss(params, state, control, extra_variables, rng_key, reg_loss_args, m_drift, m_noise)
)

# Let's jit the map gradient function
jit_gradient_update = jax.jit(
    lambda params, opt_state, state, control, extra_variables, rng_key : \
    map_gradient_update_params(params, opt_state, state, control, extra_variables, rng_key, jit_model_loss, opt)
)

In [ ]:
def evaluation_metrics(
    params : Dict[str, Any],
    key : jnp.ndarray,
    test_data : Dict[str, Any]
) -> Dict[str, float]:
    res = {}
    for n_iter in tqdm(range(num_test_batches), leave=False):
        test_batch_data = pick_batch_transitions_as_array(
            test_data, test_batch, stepsize_range_test, horizon_test, 
            problem_config_for_dataset_extraction, action_sampling_strategy
        )
        key, key_loss = jax.random.split(key)
        _, losses = jit_model_loss(params, *test_batch_data, key_loss)
        if len(res) == 0:
            res = {k: np.zeros(num_test_batches) for k in losses.keys()}
        for _key, v in losses.items():
            res[_key][n_iter] = v
    return {k: np.mean(v) for k, v in res.items()}

In [ ]:
import matplotlib
matplotlib.use("Qt5Agg")  # Or another interactive backend like "TkAgg", Qt5Agg
# matplotlib.use("widget")  # Or another interactive backend like "TkAgg", Qt5Agg
import matplotlib.pyplot as plt

In [ ]:
# Create a figure with two rows of subplots
fig, (ax1, ax2, ax3, ax4) = plt.subplots(nrows=4, ncols=1, figsize=(10, 10))

# Create placeholders for lines so we can update them later
# For each subplot, create two lines - one for ground truth and one for prediction
gt_line1, = ax1.plot([], [], 'b-', label="Ground Truth")
pred_line1, = ax1.plot([], [], 'r--', label="Prediction")

gt_line2, = ax2.plot([], [], 'b-', label="Ground Truth")
pred_line2, = ax2.plot([], [], 'r--', label="Prediction")

# Let's add a third axil with throttle
control_line = ax3.plot([], [], 'g-', label="Throttle")[0]
gear_ratio = ax3.plot([], [], 'r-', label="Gear ratio")[0]

wheelspeed_line = ax4.plot([], [], 'b-', label="Wheelspeed")[0]
wheelspeed_line_est = ax4.plot([], [], 'r--', label="Estimated")[0]

# Lists to store multiple prediction lines (will be populated later)
pred_lines1 = []
pred_lines2 = []
pred_lines3 = []
pred_lines4 = []

# Add labels and legends
ax1.set_title("Engine speed")
# ax1.set_xlabel("Time")
ax1.legend()
ax1.grid()

ax2.set_title("Boost pressure")
# ax2.set_xlabel("Time")
ax2.legend()
ax2.grid()

ax3.set_title("Throttle")
# ax3.set_xlabel("Time")
ax3.legend()
ax3.grid()

ax4.set_title("Rear wheelspeed")
ax4.set_xlabel("Time")
ax4.legend()
ax4.grid()


fig.tight_layout()
fig.show()

In [ ]:
def update_plots(time, gt_state, pred_state, controls, alpha=0.3):
    # Split the ground truth and prediction into two states
    gt_state1, gt_state2, gt_state3 = gt_state[:, 0], gt_state[:, 1], gt_state[:, 2]
    pred_state1, pred_state2, pred_state3, pred_state4 = pred_state[:, :, 0], pred_state[:, :, 1], pred_state[:, :, 2], pred_state[:, :, 3]
    # Update ground truth lines
    gt_line1.set_xdata(time)
    gt_line1.set_ydata(gt_state1)
    
    gt_line2.set_xdata(time)
    gt_line2.set_ydata(gt_state2)
    
    control_line.set_xdata(time[:-1])
    control_line.set_ydata(controls[:, 0])

    wheelspeed_line.set_xdata(time)
    wheelspeed_line.set_ydata(gt_state3)
    # wheelspeed_line_est.set_xdata(time[:-1])
    # wheelspeed_line_est.set_ydata(np.mean(est_wheel_speed, axis=0))

    # Clear previous prediction lines
    for line in pred_lines1:
        line.remove()
    pred_lines1.clear()
    
    for line in pred_lines2:
        line.remove()
    pred_lines2.clear()
    
    for line in pred_lines3:
        line.remove()
    pred_lines3.clear()

    for line in pred_lines4:
        line.remove()
    pred_lines4.clear()
    
    # Add new prediction lines for each trajectory
    n_trajectories = pred_state1.shape[0]
    for i in range(n_trajectories):
        # For state 1
        line, = ax1.plot(time, pred_state1[i], 'r-', alpha=alpha, label="Prediction" if i == 0 else "_nolegend_")
        pred_lines1.append(line)
        
        # For state 2
        line, = ax2.plot(time, pred_state2[i], 'r-', alpha=alpha, label="Prediction" if i == 0 else "_nolegend_")
        pred_lines1.append(line)
        
        # For estimated wheelspeed
        line, = ax4.plot(time, pred_state3[i], 'r-', alpha=alpha, label="Prediction" if i == 0 else "_nolegend_")
        pred_lines3.append(line)
    
        # For gear ratio
        line, = ax3.plot(time, pred_state4[i], 'r-', alpha=alpha, label="Prediction" if i == 0 else "_nolegend_")
        pred_lines4.append(line)
        
    # # Add mean prediction if there are multiple trajectories
    # if n_trajectories > 1:
    #     mean_pred1 = pred_state1.mean(axis=0)
    #     mean_pred2 = pred_state2.mean(axis=0)
        
    #     line, = ax1.plot(time, mean_pred1, 'r-', linewidth=2, label="Mean Prediction")
    #     # pred_line1.append(line)
        
    #     line, = ax2.plot(time, mean_pred2, 'r-', linewidth=2, label="Mean Prediction")
    #     # pred_line2.append(line)
    
    # # Update legends
    # ax1.legend()
    # ax2.legend()
    
    # Rescale axes
    ax1.relim()
    ax1.autoscale_view()
    ax2.relim()
    ax2.autoscale_view()
    ax3.relim()
    ax3.autoscale_view()
    ax4.relim()
    ax4.autoscale_view()
    
    # Refresh the figure
    fig.canvas.draw()
    fig.canvas.flush_events()

In [ ]:
@partial(jax.jit, static_argnames=("num_samples", "return_full_state", "num_inter_in"))
def prediction_paths(
    init_state,
    controls,
    times,
    rng_key,
    hist_states,
    params,
    num_samples = 4,
    return_full_state = False,
    num_inter_in = 1
):
    # Initialize the state and control
    rng_key = jax.random.split(rng_key, num_samples)
    vmap_integrate = jax.vmap(integrate_path, in_axes=(None, None, None, 0, None, None, None, None, None, None))
    # Integrate the dynamics for each sample
    paths, extras = vmap_integrate(
        init_state, controls, times, rng_key, params, hist_states, m_drift, m_noise, return_full_state, num_inter_in
    )
    return paths, extras

def generate_examples(
    m_params, dataset, rng_key, num_samples,
    stepsize_range = stepsize_range_test, horizon = horizon_test
):
    # Get the batch
    test_batch_data = pick_batch_transitions_as_array(
        dataset, 1, stepsize_range, horizon, 
        problem_config_for_dataset_extraction, action_sampling_strategy
    )
    # Extract the state and control
    states, controls, *_ = test_batch_data
    states = states[0]
    controls = controls[0]
    _time = states[:, -1]
    states = states[:, :-1]

    # Let's remove the split the history
    (hist_state, hist_control), (states, controls) = m_drift.split_state_with_history(states, controls)
    _time_hist = _time[:hist_state.shape[0]]
    hist_states = (hist_state, hist_control, _time_hist)
    _time = _time[-states.shape[0]:]
    # Get the initial state
    preds, _extras = prediction_paths(
        states[0], controls, _time, rng_key, hist_states, m_params, num_samples, return_full_state = True
    )
    # return _time, states, preds, controls, {"wr": wr, "est": est_wheel_speed}
    return np.array(_time), np.array(states), np.array(preds), \
        np.array(controls), { k : np.array(v) for k, v in _extras.items()}
    
    

In [ ]:
# Let's try some plotting
plot_num_samples = 4
plot_dataset = train_data
plot_rng_key = None
if plot_rng_key is None:
    plot_rng_key = jax.random.PRNGKey(42)
plot_rng_key, plot_rng_key_test = jax.random.split(plot_rng_key)
plot_stepsize_range = stepsize_range_test
plot_horizon = horizon_test
# plot_stepsize_range = (1,1)
# plot_horizon = 500
plot_freq = 20

plot_time, plot_gt_state, plot_pred_state, plot_controls, _others = generate_examples(
    my_params, plot_dataset, plot_rng_key_test, plot_num_samples,
    stepsize_range = plot_stepsize_range, horizon = plot_horizon
)

# Plot the first example
update_plots(plot_time, plot_gt_state, plot_pred_state, plot_controls, alpha=0.3)

In [ ]:
_params

In [ ]:

# Now let's prepare for the training.
save_name = "test"
rng_key = jax.random.PRNGKey(_seed)

# Let's create the checkpoint manager
track_n_checkoints = {
    "max_to_keep": 5,
    "async_exec": False,
    "metrics": {
        "Train/mse": 1.0,
        "Test/mse": 2.0
    }
}

# First let's create the checkpoint manager
ckpt_model = TrainCheckpoints(
    TRAINING_LOGS_DIR,
    save_name,
    track_n_checkoints,
    best_mode = 'min',
    writer_on = True,
    extra_config_to_save_as_yaml=model_config,
    saving_freq=save_freq
)

# Evaluate through the number of epochs
grad_step = 0
_params = my_params
for _ in tqdm(range(num_epochs)):
    # Shuffle the data indexes
    if do_iterate_through_all:
        np.random.shuffle(traj_indexes)
    
    for n_batch in tqdm(range(num_batches), leave=False):
        if do_iterate_through_all:
            train_batch_data = load_batch(traj_indexes, n_batch)
        else:
            train_batch_data = pick_batch_transitions_as_array(
                train_data, train_batch, stepsize_range, horizon, 
                problem_config_for_dataset_extraction, action_sampling_strategy
            )
            
        # Check if it's time for testing
        if grad_step % test_freq == 0:
            rng_key, rng_test_key = jax.random.split(rng_key)
            eval_metrics = evaluation_metrics(_params, rng_test_key, test_data)
            eval_metrics = {
                f"Test/{k}" : v for k, v in eval_metrics.items()
            }
        # Apply gradient updates
        rng_key, rng_grad_key = jax.random.split(rng_key)
        _params, opt_state, train_metrics = jit_gradient_update(
            _params, opt_state, *train_batch_data, rng_grad_key
        )
        train_metrics = {
            f"Train/{k}" : v for k, v in train_metrics.items()
        }
        grad_step += 1

        # Let's plot if needed
        if grad_step % plot_freq == 0:
            rng_key, plot_rng_key = jax.random.split(rng_key)
            plot_time, plot_gt_state, plot_pred_state, plot_controls, _others = generate_examples(
                _params, plot_dataset, plot_rng_key, plot_num_samples,
                stepsize_range = plot_stepsize_range, horizon = plot_horizon
            )
            update_plots(plot_time, plot_gt_state, plot_pred_state, plot_controls, alpha=0.3)

        # Save metrics
        metrics_save = {**train_metrics, **eval_metrics}
        save_dict = {
            "sde_params": _params,
            "train_metrics": train_metrics,
            "test_metrics": eval_metrics,
            "current_step": grad_step
        }
        # Write the checkpoint
        ckpt_model.write_checkpoint_and_log_data(save_dict, metrics_save)

In [ ]:
train_metrics

In [ ]:
train_metrics

In [ ]:
train_batch_data

In [ ]:
# load_model_from_config(model_config)

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

In [ ]:
# # Let's do some plotting to test
# data = result["trajectories"][2]
# time = data.get("time", list(range(len(next(iter(data.values()))))))  # Fallback if time is missing

# quantity_mapping = {
#     "Throttle": "throttle",
#     "Combined Rear Wheelspeed": "rear_wheelspeed",
#     "Engine Speed": "engine_speed",
#     "Boost Pressure": "boost_pressure",
#     "Brake": "brake",
#     "Filter Acceleration": "accel_x_filtered",
#     "Wheel prop torque": "wheelRR_propTorque",
#     # "Parking Brake" : "pbrake",
#     # "Realistic" : "driveStatus_isRealisticDrive"
# }

# fig, axes = plt.subplots(len(quantity_mapping), sharex=True, figsize=(10, 12))

# for ax, (label, key) in zip(axes, quantity_mapping.items()):
#     if key in data:
#         ax.plot(time, data[key])
#         ax.set_title(label)
#         ax.grid()
#     else:
#         ax.text(0.5, 0.5, f"{key} not found", ha="center", transform=ax.transAxes)
#         ax.set_title(label)
        
# axes[-1].set_xlabel("Time")
# plt.tight_layout()
# plt.show()

In [ ]:
# # Let's do some plotting to test
# data = result["trajectories"][0]
# time = data.get("time", list(range(len(next(iter(data.values()))))))  # Fallback if time is missing

# quantity_mapping = {
#     "Throttle": "throttle",
#     "Combined Rear Wheelspeed": "rear_wheelspeed",
#     "Engine Speed": "engine_speed",
#     "Boost Pressure": "boost_pressure",
#     "Brake": "brake",
#     "Filter Acceleration": "accel_x_filtered",
#     "Parking Brake" : "pbrake",
#     "Realistic" : "driveStatus_isRealisticDrive"
# }

# fig, axes = plt.subplots(len(quantity_mapping), sharex=True, figsize=(10, 12))

# for ax, (label, key) in zip(axes, quantity_mapping.items()):
#     if key in data:
#         ax.plot(time, data[key])
#         ax.set_title(label)
#         ax.grid()
#     else:
#         ax.text(0.5, 0.5, f"{key} not found", ha="center", transform=ax.transAxes)
#         ax.set_title(label)
        
# axes[-1].set_xlabel("Time")
# plt.tight_layout()
# plt.show()